# Aula 1 — Git e versionamento
### DS-PY-004 · Técnicas de Programação I

Hoje:

1. Por que versionar
2. Os três estados do Git
3. Primeiro repositório
4. Ver e desfazer mudanças
5. `.gitignore`
6. GitHub: publicar, trazer, clonar
7. Branches

> Os comandos abaixo aparecem com `!` porque estamos no Jupyter. **No seu terminal, rode sem o `!`.**

> **Windows, macOS ou Linux?** Os comandos `git` são idênticos nos três. O que muda são os comandos de sistema (`ls`, `mkdir`) — por isso aqui usamos Python no lugar deles, que funciona igual em qualquer máquina.
>
> No Windows, você pode rodar o `git` tanto no **PowerShell** quanto no **Git Bash** (instalado junto com o Git). O Git Bash entende os comandos de Linux (`ls`, `pwd`, `cat`), então se algum tutorial da internet usar `ls`, é nele que vai funcionar.

## 1. Por que versionar

Como você salva versões de um trabalho hoje?

```
analise.ipynb
analise_v2.ipynb
analise_v2_FINAL.ipynb
analise_v2_FINAL_agora_vai.ipynb
```

Problemas: qual é a boa? o que mudou entre elas? como três pessoas trabalham juntas? como volto ao que funcionava ontem?

**Git resolve os quatro.** Ele grava fotografias do projeto ao longo do tempo, sabe exatamente o que mudou entre uma e outra e sabe juntar o trabalho de várias pessoas.

Em Data Science isso pesa ainda mais: análise é trabalho experimental. Você testa uma abordagem, não gosta, testa outra — e precisa saber como chegou no número final.

In [1]:
!git --version

git version 2.39.5


In [2]:
# Helper multiplataforma: substitui o `ls` do Linux/Mac, que não existe no Windows
from pathlib import Path

def listar(pasta='.', ocultos=False):
    """Lista o conteúdo de uma pasta. Funciona igual no Windows, macOS e Linux."""
    itens = sorted(
        p.name + ('/' if p.is_dir() else '')
        for p in Path(pasta).iterdir()
        if ocultos or not p.name.startswith('.')
    )
    print('  '.join(itens) if itens else '(pasta vazia)')

## 2. Os três estados

```
   WORKING DIRECTORY        STAGING AREA           REPOSITÓRIO
   (sua pasta)              (a "sacola")           (o histórico)

   analise.py   ──git add──▶  analise.py  ──git commit──▶  ┌─────────┐
   dados.csv                                               │ commit3 │
   README.md                                               ├─────────┤
                                                           │ commit2 │
   você edita aqui          você escolhe o que             ├─────────┤
                            vai no próximo commit          │ commit1 │
                                                           └─────────┘
```

**Por que dois passos?** Porque você quase nunca quer commitar tudo que mudou. Mexeu em 5 arquivos, mas só 2 fazem parte da mesma ideia. O `add` deixa você montar um commit que conta **uma história coerente**.

## 3. Configuração e primeiro repositório

Todo commit é assinado. Sem nome e e-mail configurados, o Git se recusa a commitar.

In [4]:
!git config --global user.name "danielmassita"
!git config --global user.email "daniel.massita@gmail.com"
!git config --global init.defaultBranch main

!git config --list --global

user.name=danielmassita
user.email=daniel.massita@gmail.com
init.defaultbranch=main


`--global` = vale para todos os seus repositórios. Sem ele, vale só para o repositório atual.

In [9]:
import os
os.makedirs('demo-aula1', exist_ok=True)
%cd demo-aula1
!git init

/home/usuario/Downloads/Técnicas de Programação I (PY) - 1 - 24 de set/demo-aula1/demo-aula1/demo-aula1
Initialized empty Git repository in /home/usuario/Downloads/Técnicas de Programação I (PY) - 1 - 24 de set/demo-aula1/demo-aula1/demo-aula1/.git/


In [10]:
listar(ocultos=True)

.git/


A pasta `.git` é o que transforma uma pasta comum em repositório. Todo o histórico vive lá dentro. Apagou a pasta, apagou o histórico.

In [11]:
!git status

On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


`git status` é o comando que você mais vai usar. Ele diz o estado atual **e sugere o próximo comando**. Aprenda a lê-lo e você se vira sozinho.

## 4. O ciclo: add → commit

In [23]:
with open('README.md', 'w', encoding='utf-8') as f:
    f.write('# Projeto de demonstração\n\nAula 1 do módulo DS-PY-004.\n')

!git status

On branch main
nothing to commit, working tree clean


`Untracked files`: o arquivo existe, mas o Git ainda não recebeu ordem de tomar conta dele.

In [24]:
!git add README.md
!git status

On branch main
nothing to commit, working tree clean


Agora `Changes to be committed` — está na sacola.

In [25]:
!git commit -m "do  o"

On branch main
nothing to commit, working tree clean


### Mensagens de commit

| ❌ | ✅ |
|---|---|
| `update` | `fix: corrige calculo de media ignorando nulos` |
| `ajustes` | `feat: adiciona leitura de arquivos parquet` |
| `commit` | `docs: documenta parametros da funcao de limpeza` |

Uma boa mensagem completa a frase: *"Se aplicado, este commit vai..."*

In [26]:
!git log --oneline

b767945 (HEAD -> main) do  o


## 5. Ver e desfazer mudanças

In [29]:
with open('README.md', 'a', encoding='utf-8') as f:
    f.write('\n## Objetivo\n\nDemonstrar o fluxo básico de versionamento.\n')

!git diff

diff --git a/README.md b/README.md
index c28944a..a032feb 100644
--- a/README.md
+++ b/README.md
@@ -1,3 +1,15 @@
 # Projeto de demonstração
 
 Aula 1 do módulo DS-PY-004.
+
+## Objetivo
+
+Demonstrar o fluxo básico de versionamento.
+
+## Objetivo
+
+Demonstrar o fluxo básico de versionamento.
+
+## Objetivo
+
+Demonstrar o fluxo básico de versionamento.


`-` é o que saiu, `+` é o que entrou. É isso que o Git guarda: não uma cópia inteira a cada versão, mas a diferença.

Depois do `add`, use `git diff --staged`.

In [30]:
!git add README.md
!git commit -m "docs: detalha objetivo do projeto"

[main d80a045] docs: detalha objetivo do projeto
 1 file changed, 12 insertions(+)


In [32]:
# mudei de ideia sobre uma alteração ainda não commitada
with open('README.md', 'a', encoding='utf-8') as f:
    f.write('\nUMA ALTERAÇÃO DA QUAL ME ARREPENDI\n')
!git add README.md
!git restore --staged README.md
!git status

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   README.md

no changes added to commit (use "git add" and/or "git commit -a")


> ⚠️ `restore` **descarta** a alteração. Não tem desfazer do desfazer.

## 6. `.gitignore`

In [34]:
import os
os.makedirs('dados', exist_ok=True)
open('credenciais.txt', 'w').write('senha=123456')
open('dados/base_gigante.csv', 'w').write('a,b\n1,2\n')

!git status --short

 M README.md
?? credenciais.txt
?? dados/


### O que NÃO deve entrar no repositório

| Não commitar | Por quê |
|---|---|
| Credenciais, tokens, `.env` | Ficam no histórico **para sempre**. Token commitado = token comprometido |
| Bases de dados grandes | O histórico guarda todas as versões: o repositório incha e não desincha |
| `.ipynb_checkpoints/`, `__pycache__/` | Lixo da ferramenta, não é o seu trabalho |
| `venv/`, `node_modules/` | Reinstaláveis a partir das dependências |

In [35]:
with open('.gitignore', 'w', encoding='utf-8') as f:
    f.write('*.txt\ndados/\n.ipynb_checkpoints/\n__pycache__/\n.env\n')

!git status --short

 M README.md
?? .gitignore


In [36]:
!git add .gitignore
!git commit -m "chore: adiciona gitignore"

[main 38ff9bc] chore: adiciona gitignore
 1 file changed, 5 insertions(+)
 create mode 100644 .gitignore


> ⚠️ O `.gitignore` só vale para arquivos **ainda não rastreados**. Se você já commitou, é preciso `git rm --cached <arquivo>` antes.

## 7. GitHub

```
   SUA MÁQUINA                          GITHUB
   ┌──────────────┐    git push    ┌──────────────┐
   │  repositório │ ─────────────▶ │  repositório │
   │    local     │ ◀───────────── │    remoto    │
   └──────────────┘    git pull    └──────────────┘
                                          │ git clone
                                          ▼
                                   ┌──────────────┐
                                   │  máquina do  │
                                   │   colega     │
                                   └──────────────┘
```

Git é **distribuído**: cada cópia tem o histórico completo. O GitHub não é "o Git" — é um lugar combinado onde todo mundo sincroniza.

In [ ]:
!git remote add origin https://github.com/leosi123/demo-aula1.git
!git remote -v
!git push -u origin main

origin	https://github.com/leosi123/demo-aula1.git (fetch)
origin	https://github.com/leosi123/demo-aula1.git (push)
Username for 'https://github.com': 

> ⚠️ **`git remote add` só funciona uma vez.** Se você rodar esta célula de novo, verá `error: remote origin already exists` — o Git recusa criar um segundo remoto com o mesmo nome e **mantém o antigo**. Se a URL tiver mudado, o push continua indo para o endereço velho.
>
> Por isso a célula abaixo verifica antes: usa `add` na primeira vez e `set-url` (que **troca** o endereço) nas seguintes.

**O GitHub não aceita a senha da conta no terminal.** Quando pedir `Password`, cole um **token de acesso**:

Foto do perfil → **Settings** → **Developer settings** → **Personal access tokens** → **Tokens (classic)** → **Generate new token (classic)** → escopo **`repo`**.

> Ao colar o token no terminal **nada aparece na tela**. Nem asteriscos. Cole e aperte Enter.

O `-u` amarra a branch local à remota. É só na primeira vez; depois basta `git push`.

In [ ]:
!git pull origin main
!git log --oneline

| | o que faz |
|---|---|
| `git fetch` | baixa as novidades, **não** mexe nos seus arquivos |
| `git pull` | `fetch` + `merge`: baixa e já aplica |

Conferir a caixa de correio vs. abrir as cartas e guardar.

O **histórico inteiro** veio junto — é o que diferencia clonar de baixar um .zip.

## 8. Branches

Você quer testar uma abordagem diferente sem estragar o que já funciona.

```
                      ┌── C4 ── C5 ──┐  feature/estatisticas
                      │              ▼
   C1 ── C2 ── C3 ────┴──────────────C6   main
                                     ▲
                                   merge
```

Uma branch é uma **linha do tempo paralela**. Deu certo, funde na `main`; deu errado, apaga e nada aconteceu.

In [ ]:
!git switch -c feature/estatisticas
!git branch

In [ ]:
with open('estatisticas.py', 'w', encoding='utf-8') as f:
    f.write('def media(valores):\n    return sum(valores) / len(valores)\n')

!git add estatisticas.py
!git commit -m "feat: adiciona funcao de media"
listar()

In [ ]:
!git switch main
listar()

**O arquivo sumiu da pasta.** Nada foi perdido — ele está na outra linha do tempo. Os arquivos na sua tela mudam conforme a branch em que você está.

In [ ]:
!git switch feature/estatisticas
listar()

In [ ]:
!git switch main
!git merge feature/estatisticas
!git branch -d feature/estatisticas
listar()

In [ ]:
!git log --oneline --graph --all

In [ ]:
!git push

### Conflitos

Se duas pessoas mudarem a **mesma linha** do **mesmo arquivo** em branches diferentes, o Git não adivinha. Ele para e marca o trecho:

```
<<<<<<< HEAD
media = soma / total
=======
media = np.mean(valores)
>>>>>>> feature/estatisticas
```

Você edita deixando como deve ficar, apaga os marcadores, `git add`, `git commit`.

---
## Cola de sobrevivência

```bash
git init                  # cria o repositório
git status                # onde eu estou? (rode SEMPRE)
git add <arquivo>         # coloca na sacola
git commit -m "mensagem"  # grava no histórico
git log --oneline         # o que já foi feito
git diff                  # o que mudou e não foi para a sacola
git restore <arquivo>     # descarta a alteração

git clone <url>           # baixa um repositório
git push                  # envia para o GitHub
git pull                  # traz do GitHub

git switch -c <nome>      # cria e entra numa branch
git switch <nome>         # troca de branch
git merge <nome>          # traz a branch para a atual
```

> **Commit é barato. Commite cedo, commite sempre.**

## Agora é com você

Abra `aula1_git.ipynb`.